# Experiment No. 09
## Development of a Personalized AI Assistant using an Open-Source LLM, RAG, and LoRA Fine-Tuning

**Aim:** To develop a personalized AI assistant using an open-source language model, integrate Retrieval-Augmented Generation (RAG) to answer questions from user-specific documents, and investigate the effect of LoRA fine-tuning on response quality.

**Note:** Run cells in order. Select a GPU runtime in Google Colab if available. This notebook has been prepared but has not been executed in your Colab session. Record actual outputs and results from your run.


## 1. Install required libraries

In [1]:
!pip -q install transformers peft accelerate datasets sentence-transformers pypdf


## 2. Load a pretrained open-source LLM

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype
).to(device)

print("Model loaded successfully!")
print("Device:", device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda


## 3. Generate a response before fine-tuning

In [3]:
def generate_answer(active_model, question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    active_model.eval()
    with torch.no_grad():
        outputs = active_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(generate_answer(base_model, "Explain artificial intelligence in simple terms."))


Artificial Intelligence (AI) is a field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and understanding natural language. AI involves the development of algorithms and machine learning techniques to enable computers to learn from data, recognize patterns, and make decisions based on those patterns.

In simpler terms, AI refers to the ability of machines to think like humans and perform tasks that usually require human intelligence. It includes areas such as machine learning, natural language processing, robotics, autonomous vehicles, and more. AI aims to create systems that can interact with people in ways that mimic human behavior, making them more efficient, effective, and useful for various applications.


## 4. Upload a text-based PDF document for RAG

In [4]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
if not pdf_files:
    raise ValueError("Please upload a text-based PDF file.")

pdf_path = pdf_files[0]
reader = PdfReader(pdf_path)
document_text = "\n".join(page.extract_text() or "" for page in reader.pages)

if not document_text.strip():
    raise ValueError("No selectable text found. Try a text-based PDF rather than a scanned image PDF.")

print("Uploaded:", pdf_path)
print("Pages:", len(reader.pages))
print("Characters extracted:", len(document_text))


Saving TRUST-INDEX.pdf to TRUST-INDEX.pdf
Uploaded: TRUST-INDEX.pdf
Pages: 7
Characters extracted: 3962


## 5. Split document into chunks

In [5]:
def split_text(text, chunk_size=700, overlap=100):
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = split_text(document_text)
print("Chunks created:", len(chunks))
print("Sample chunk:", chunks[0][:500] if chunks else "No chunks found")


Chunks created: 7
Sample chunk: TRUST Index
A Trust-Aware Multi-Agent Framework for
Hallucination Detection and Verification in
Large Language Models

LLMs can generate confident and convincing responses that may be false
or unsupported.
RAG reduces hallucinations but cannot guarantee the correctness of
generated responses.
Errors can still occur due to incorrect retrieval, conflicting evidence,
reasoning, or outdated information.
Existing approaches such as RAG, GraphRAG, CRAG, Self-RAG and ReAct
have limited independent veri


## 6. Create embeddings and retrieve relevant chunks

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

document_embeddings = embedding_model.encode(
    chunks,
    convert_to_tensor=True,
    normalize_embeddings=True
)

def retrieve_documents(question, top_k=3):
    query_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    scores = document_embeddings @ query_embedding
    k = min(top_k, len(chunks))
    indices = torch.topk(scores, k=k).indices.tolist()
    return [chunks[i] for i in indices]

print("\n".join(retrieve_documents("What is the main topic of this document?")))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

without replacing them.
Data Collection
Collect PDF, DOCX, and TXT documents as the system's knowledge base.
Generate questions from the collected documents.
Process questions using different RAG approaches.
Test responses using open-source LLMs.
Store generated responses and retrieved document chunks.
Store supporting evidence and model outputs for analysis.
Test different embedding models and LLM combinations.
Compare results to identify the most suitable combination for the proposed
framework.
Data Processing
Convert collected PDF, DOCX, and TXT documents into clean text.
Remove unnecessary formatting, noise, and irrelevant content.
Divide text into meaningful chunks with suitable overla
y formatting, noise, and irrelevant content.
Divide text into meaningful chunks with suitable overlap.
Generate vector embeddings using Sentence Transformers.
Store embeddings in FAISS/ChromaDB.
Attach metadata such as document name, page number, and chunk ID.
Prepare questions and generated respons

## 7. Generate answers using RAG

In [7]:
def rag_answer(question):
    retrieved_chunks = retrieve_documents(question)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using only the context below.
If the answer is not available in the context, say that the document does not provide it.

Context:
{context}

Question: {question}
Answer:"""

    return generate_answer(base_model, prompt, max_new_tokens=180)

question = "What are the main points discussed in my document?"
print(rag_answer(question))


Based on the provided context, there is no specific mention of what is being discussed in your document. Therefore, I cannot provide any answers or generate a response based solely on the given information. The document does not contain enough details to determine the main points discussed.


## 8. Prepare a small demonstration dataset for LoRA

In [8]:
from datasets import Dataset

training_examples = [
    {"text": "User: What is RAG?\nAssistant: RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and provides it to a language model as context for a grounded answer."},
    {"text": "User: What is LoRA?\nAssistant: LoRA is a parameter-efficient fine-tuning technique that trains small additional matrices while keeping the original model weights frozen."},
    {"text": "User: What is an embedding?\nAssistant: An embedding is a numerical vector representation of text that helps a system compare semantic similarity."},
    {"text": "User: What is a vector database?\nAssistant: A vector database stores vector representations and helps retrieve semantically similar information."},
    {"text": "User: What is an LLM?\nAssistant: A Large Language Model is an AI model trained on large amounts of text to understand and generate language."},
    {"text": "User: Why use fine-tuning?\nAssistant: Fine-tuning adapts a pretrained model to a specific task, response style, or collection of examples."}
]

train_dataset = Dataset.from_list(training_examples)
print("Training examples:", len(train_dataset))


Training examples: 6


## 9. Apply LoRA

In [11]:
import subprocess
import sys

# Ensure a compatible torchao version is installed to prevent PEFT compatibility errors
try:
    import torchao
    from packaging import version
    if version.parse(torchao.__version__) < version.parse("0.16.0"):
        print("Upgrading torchao to avoid compatibility issues with PEFT...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-U", "torchao"], check=True)
except ImportError:
    # If torchao is not installed, standard PEFT setup will proceed without it
    pass

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

Upgrading torchao to avoid compatibility issues with PEFT...
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 10. Tokenize data and fine-tune

In [12]:
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora_assistant_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=(device == "cuda")
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

train_result = trainer.train()
print("LoRA fine-tuning completed.")
print("Training loss:", train_result.training_loss)


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Step,Training Loss
1,3.404976
2,2.807769
3,3.532771
4,2.685376
5,3.309248
6,3.377844
7,2.505771
8,3.266829
9,3.118729


LoRA fine-tuning completed.
Training loss: 3.1121459007263184


## 11. Compare responses before and after fine-tuning

In [13]:
test_questions = [
    "What is RAG?",
    "What is LoRA?",
    "What is an embedding?"
]

for question in test_questions:
    print("\nQuestion:", question)
    print("Before fine-tuning:", generate_answer(base_model, question))
    print("After fine-tuning:", generate_answer(lora_model, question))



Question: What is RAG?
Before fine-tuning: RAG stands for "Reinforcement Learning Agent." It's a type of artificial intelligence (AI) that uses reinforcement learning to learn from its environment and improve its performance over time.

In simple terms, RAG agents are self-learning systems that can learn to solve complex problems on their own without being explicitly programmed or taught. They use trial-and-error methods to find the best way to interact with their environment, which helps them adapt to new situations and improve their performance over time.

RAGs are often used in fields such as robotics, autonomous vehicles, and game AI, where they need to make decisions based on feedback from their environment rather than pre-programmed rules. The goal is to create agents that can learn from experience and become more efficient at
After fine-tuning: RAG stands for "Reinforcement Learning Agent." It's a type of artificial intelligence (AI) that uses reinforcement learning to learn fr

## 12. Save the LoRA adapter

In [14]:
adapter_path = "./lora_adapter"
lora_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved to:", adapter_path)


LoRA adapter saved to: ./lora_adapter


## Observations

1. The pretrained LLM generated responses to user queries.
2. RAG retrieved relevant text chunks from the uploaded document and supplied them as context.
3. LoRA trained a small set of additional parameters while the base model weights remained frozen.
4. Responses before and after fine-tuning can be compared for relevance, clarity, and response style.

## Results

A personalized AI assistant workflow was implemented using an open-source LLM, document retrieval, and LoRA fine-tuning. Record the actual outputs from your run. Improvement in response quality is not guaranteed with the small demonstration dataset.

## Conclusion

The experiment demonstrates how an open-source LLM can answer questions using user-specific documents through RAG and how LoRA can adapt a model efficiently. The generated responses can be compared before and after fine-tuning to investigate its effect on response quality.

## Troubleshooting

- **GPU unavailable:** The notebook can run on CPU, but generation and fine-tuning may be slower.
- **PDF has no extracted text:** Upload a text-based PDF rather than a scanned image.
- **Out-of-memory error:** Restart the runtime, select a GPU with more memory if available, or reduce `max_length`, batch size, and generation tokens.
- **Fine-tuning comparison:** Use the same test questions before and after training. A small demonstration dataset may not produce measurable improvements.
